# Fruit & Vegetable Quality Grading System using Deep CNN
### End-to-End Computer Vision & Deep Learning Project

**Objective**: Automatically inspect, detect, and assign an industrial quality grade (Grade A, Grade B, Grade C) to fruit and vegetable samples using a custom Deep Convolutional Neural Network (CNN) and Explainable AI (Grad-CAM).

---

## 1. Environment Setup and Library Imports

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Local project modules
from models.cnn_model import FruitQualityCNN, build_model
from models.gradcam import GradCAM
from models.quality_grader import QualityGrader
from utils.transforms import get_train_transforms, get_val_transforms, preprocess_image_for_model
from utils.dataset_generator import create_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

## 2. Dataset Preparation & Augmentation
We load data organized into standard ImageFolder splits with data augmentations (flips, rotations, jitter, affine shifts).

In [ ]:
# Ensure dataset exists
if not os.path.exists("data/dataset/train"):
    create_dataset(output_dir="data/dataset", train_per_class=60, val_per_class=20)

train_tf = get_train_transforms((128, 128))
val_tf = get_val_transforms((128, 128))

train_ds = ImageFolder("data/dataset/train", transform=train_tf)
val_ds = ImageFolder("data/dataset/val", transform=val_tf)

class_names = train_ds.classes
print(f"Target Classes ({len(class_names)}): {class_names}")
print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")

## 3. Explore Dataset Samples

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for idx, cls_name in enumerate(class_names):
    ax = axes[idx // 4, idx % 4]
    sample_file = f"data/samples/{cls_name}_test.png"
    if os.path.exists(sample_file):
        img = Image.open(sample_file)
        ax.imshow(img)
        ax.set_title(cls_name.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.axis('off')
plt.suptitle("Specimen Visual Samples (Fresh vs Rotten Categories)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. CNN Architecture Architecture & Initialization
Our custom `FruitQualityCNN` has 4 dual-conv hierarchical stages with Batch Normalization, ReLU activations, Max Pooling, and Dropout, terminating in an Adaptive Average Pooling layer and Linear Classifier.

In [ ]:
model = build_model(num_classes=len(class_names), model_type="custom_cnn").to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Architecture: FruitQualityCNN")
print(f"Total Trainable Parameters: {total_params:,}")
print(model)

## 5. Model Inference & Industrial Quality Grading
Load trained checkpoint and evaluate quality grades.

In [ ]:
ckpt_path = "checkpoints/best_model.pth"
if os.path.exists(ckpt_path):
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print("Loaded checkpoint successfully!")

grader = QualityGrader(class_names=class_names)
gradcam = GradCAM(model)

# Test on a rotten tomato
test_image_path = "data/samples/rotten_tomato_test.png"
tensor, orig_rgb = preprocess_image_for_model(test_image_path, (128, 128), device=device)

with torch.no_grad():
    logits = model(tensor)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

heatmap_2d, _ = gradcam.generate_heatmap(tensor)
overlay_rgb = gradcam.overlay_heatmap(orig_rgb, heatmap_2d)
report = grader.grade(probabilities=probs, heatmap_2d=heatmap_2d)

print(f"Assigned Grade: {report.grade} ({report.grade_description})")
print(f"Freshness Score: {report.freshness_score}%")
print(f"Defect Coverage: {report.defect_percentage}%")
print(f"Recommended Action: {report.recommended_action}")

# Plot diagnosis
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(orig_rgb)
axes[0].set_title("Original Specimen", fontweight='bold')
axes[0].axis('off')
axes[1].imshow(overlay_rgb)
axes[1].set_title("Grad-CAM Defect Attention", fontweight='bold')
axes[1].axis('off')
plt.suptitle(f"{report.commodity}: {report.grade} ({report.condition})", fontsize=13, fontweight='bold')
plt.show()